# Reproduction: Saini et al. (2023) hybrid RF+XGBoost

Paper's methods only: 5 baselines (DT, LR, RF, XGBoost, MLP) plus the paper's RF+XGBoost averaging hybrid, with SHAP.

Set `DATASET` below to one of: `nsl_kdd`, `unsw_nb15`, `cic_ids2017`, `cse_cic_ids2018`.

Setup on Kaggle: turn Internet ON (Settings), add the dataset for `DATASET` as an input, run all cells. The code is cloned from GitHub.

In [ ]:
DATASET = "cse_cic_ids2018"   # <-- change this line only

# File patterns (searched recursively under /kaggle/input) that make up each dataset's raw files
PATTERNS = {
    "nsl_kdd": ["KDDTrain+*", "KDDTest+*"],
    "unsw_nb15": ["UNSW_NB15_training-set.csv", "UNSW_NB15_testing-set.csv"],
    "cic_ids2017": ["*WorkingHours*.csv"],
    "cse_cic_ids2018": ["*-2018.csv"],
}

In [ ]:
import glob, os, shutil, subprocess, sys

WORK = "/kaggle/working"
os.environ["RIS_DATA_RAW"] = f"{WORK}/data/raw"
os.environ["RIS_DATA_PROCESSED"] = f"{WORK}/data/processed"
os.environ["RIS_RESULTS"] = f"{WORK}/results"

# Clone (or update) the repo; needs Internet turned on in notebook settings
REPO = f"{WORK}/RIS-M26"
if os.path.isdir(REPO):
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/Sahoo-Achyutananda/RIS-M26.git", REPO], check=True)
SRC = f"{REPO}/src"
print("src:", SRC)

# Prefer prepared data (output of kaggle_prep) when it is attached as an input
prepared = glob.glob(f"/kaggle/input/**/{DATASET}/dataset_final.csv", recursive=True)
SKIP_PREP = bool(prepared)
if SKIP_PREP:
    pdest = f"{WORK}/data/processed/{DATASET}"
    os.makedirs(pdest, exist_ok=True)
    shutil.copy(prepared[0], pdest)
    print("using prepared data:", prepared[0])
else:
    # No prepared data attached: fall back to the raw files
    # Copy this dataset's raw files into the layout the pipeline expects
    dest = f"{WORK}/data/raw/{DATASET}"
    os.makedirs(dest, exist_ok=True)
    found = []
    for pat in PATTERNS[DATASET]:
        found += glob.glob(f"/kaggle/input/**/{pat}", recursive=True)
    if not found:
        for root, _, files in os.walk("/kaggle/input"):
            for fn in files:
                print(os.path.join(root, fn), os.path.getsize(os.path.join(root, fn)) // 1024, "KB")
        raise AssertionError(f"No raw files matched for {DATASET}; paste the file list above so the patterns can be fixed")
    for f in sorted(set(found)):
        shutil.copy(f, dest)
        print("copied", os.path.basename(f))

In [ ]:
!pip install -q imbalanced-learn shap xgboost

In [ ]:
proc = subprocess.run(
    [sys.executable, "-u", "run_pipeline.py", "--dataset", DATASET, "--mode", "reproduce"] + (["--skip-prep"] if SKIP_PREP else []),
    cwd=SRC, env=os.environ, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
print(proc.stdout[-6000:])
print("exit code:", proc.returncode)

In [ ]:
import pandas as pd
pd.read_csv(f"{WORK}/results/{DATASET}/metrics_reproduce.csv", index_col=0).round(4)